In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/MediScan/data")

print("Data path exists:", DATA_PATH.exists())
print("Contents:", [item.name for item in DATA_PATH.iterdir()])

Data path exists: True
Contents: ['Testing', 'Training', 'splits']


In [7]:
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/MediScan/data")

TRAIN_PATH = DATA_PATH / "Training"
TEST_PATH = DATA_PATH / "Testing"

print("Training classes:")
print(sorted([
    folder.name
    for folder in TRAIN_PATH.iterdir()
    if folder.is_dir()
]))

print("\nTesting classes:")
print(sorted([
    folder.name
    for folder in TEST_PATH.iterdir()
    if folder.is_dir()
]))

Training classes:
['glioma', 'meningioma', 'notumor', 'pituitary']

Testing classes:
['glioma', 'meningioma', 'notumor', 'pituitary']


In [8]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from pathlib import Path
import pandas as pd
import numpy as np

print("All Day 3 libraries imported successfully!")

All Day 3 libraries imported successfully!


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [11]:
NUM_CLASSES = 4
IMAGE_SIZE = 224
BATCH_SIZE = 32

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

print("Number of classes:", NUM_CLASSES)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Classes:", CLASS_NAMES)

Number of classes: 4
Image size: 224
Batch size: 32
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [12]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load ImageNet-pretrained EfficientNet-B0
weights = EfficientNet_B0_Weights.DEFAULT

efficientnet = efficientnet_b0(weights=weights)

print("EfficientNet-B0 loaded successfully!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 76.2MB/s]


EfficientNet-B0 loaded successfully!


In [13]:
print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [14]:
efficientnet.classifier[1] = nn.Linear(
    efficientnet.classifier[1].in_features,
    NUM_CLASSES
)

print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


In [15]:
for param in efficientnet.features.parameters():
    param.requires_grad = False

print("EfficientNet feature layers frozen.")

EfficientNet feature layers frozen.


In [16]:
trainable_params = sum(
    p.numel()
    for p in efficientnet.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in efficientnet.parameters()
)

frozen_params = total_params - trainable_params

print("Total parameters     :", total_params)
print("Trainable parameters :", trainable_params)
print("Frozen parameters    :", frozen_params)

Total parameters     : 4012672
Trainable parameters : 5124
Frozen parameters    : 4007548


In [17]:
efficientnet = efficientnet.to(device)

print("EfficientNet-B0 moved to:", device)

EfficientNet-B0 moved to: cuda


In [19]:
from pathlib import Path

SPLIT_PATH = Path("/content/drive/MyDrive/MediScan/data/splits")

print("Split files:")
for file in SPLIT_PATH.iterdir():
    print(file.name)

Split files:
train.csv
validation.csv
test.csv


In [20]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/MediScan/data")
SPLIT_PATH = DATA_PATH / "splits"

train_df = pd.read_csv(SPLIT_PATH / "train.csv")
val_df = pd.read_csv(SPLIT_PATH / "validation.csv")
test_df = pd.read_csv(SPLIT_PATH / "test.csv")

print("Split files loaded successfully!")
print("=" * 50)

print("Training   :", len(train_df))
print("Validation :", len(val_df))
print("Testing    :", len(test_df))

print("\nTotal:", len(train_df) + len(val_df) + len(test_df))

print("\nColumns:")
print(train_df.columns.tolist())

Split files loaded successfully!
Training   : 5040
Validation : 1080
Testing    : 1080

Total: 7200

Columns:
['split', 'class', 'filename', 'path']


In [21]:
print("Training classes:")
print(sorted(train_df["class"].unique()))

print("\nValidation classes:")
print(sorted(val_df["class"].unique()))

print("\nTesting classes:")
print(sorted(test_df["class"].unique()))

print("\nClass counts:")
print("\nTraining:")
print(train_df["class"].value_counts().sort_index())

print("\nValidation:")
print(val_df["class"].value_counts().sort_index())

print("\nTesting:")
print(test_df["class"].value_counts().sort_index())

Training classes:
['glioma', 'meningioma', 'notumor', 'pituitary']

Validation classes:
['glioma', 'meningioma', 'notumor', 'pituitary']

Testing classes:
['glioma', 'meningioma', 'notumor', 'pituitary']

Class counts:

Training:
class
glioma        1260
meningioma    1260
notumor       1260
pituitary     1260
Name: count, dtype: int64

Validation:
class
glioma        270
meningioma    270
notumor       270
pituitary     270
Name: count, dtype: int64

Testing:
class
glioma        270
meningioma    270
notumor       270
pituitary     270
Name: count, dtype: int64


In [22]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


class BrainTumorDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

        self.class_names = sorted(
            self.dataframe["class"].unique()
        )

        self.class_to_idx = {
            class_name: idx
            for idx, class_name in enumerate(self.class_names)
        }

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["path"]
        class_name = row["class"]

        # Load grayscale MRI
        image = Image.open(image_path).convert("L")

        # Convert grayscale → RGB
        image = image.convert("RGB")

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        # Convert class name → numerical label
        label = self.class_to_idx[class_name]

        return image, label


print("BrainTumorDataset recreated successfully!")

BrainTumorDataset recreated successfully!


In [23]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [24]:
val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

print("Image transformations created successfully!")

Image transformations created successfully!


In [25]:
train_dataset = BrainTumorDataset(
    train_df,
    transform=train_transform
)

val_dataset = BrainTumorDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = BrainTumorDataset(
    test_df,
    transform=val_test_transform
)

print("Datasets recreated successfully!")
print("=" * 50)

print("Training dataset  :", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset   :", len(test_dataset))

print("\nClass mapping:")
print(train_dataset.class_to_idx)

Datasets recreated successfully!
Training dataset  : 5040
Validation dataset: 1080
Testing dataset   : 1080

Class mapping:
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [26]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders recreated successfully!")
print("=" * 50)

print("Training batches  :", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches   :", len(test_loader))

DataLoaders recreated successfully!
Training batches  : 158
Validation batches: 34
Testing batches   : 34


In [27]:
images, labels = next(iter(train_loader))

print("Day 3 DataLoader Verification")
print("=" * 50)

print("Images shape :", images.shape)
print("Labels shape :", labels.shape)
print("Images dtype :", images.dtype)
print("Labels dtype :", labels.dtype)

print("\nPixel range:")
print("Min:", images.min().item())
print("Max:", images.max().item())

print("\nLabels:")
print(labels)

Day 3 DataLoader Verification
Images shape : torch.Size([32, 3, 224, 224])
Labels shape : torch.Size([32])
Images dtype : torch.float32
Labels dtype : torch.int64

Pixel range:
Min: 0.0
Max: 1.0

Labels:
tensor([3, 2, 1, 2, 0, 1, 2, 2, 0, 0, 0, 2, 2, 0, 1, 3, 1, 3, 2, 2, 0, 0, 1, 3,
        2, 3, 0, 2, 2, 0, 3, 3])


In [28]:
# Move model to GPU
efficientnet = efficientnet.to(device)

# Get one batch
images, labels = next(iter(train_loader))

# Move images to GPU
images = images.to(device)

# Forward pass
with torch.no_grad():
    outputs = efficientnet(images)

print("EfficientNet-B0 Verification")
print("=" * 50)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)
print("Output dtype:", outputs.dtype)

EfficientNet-B0 Verification
Input shape : torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 4])
Output dtype: torch.float32


In [29]:
from torchvision.models import resnet50, ResNet50_Weights

# Load ImageNet-pretrained ResNet50
weights = ResNet50_Weights.DEFAULT

resnet = resnet50(weights=weights)

print("ResNet50 loaded successfully!")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 119MB/s]


ResNet50 loaded successfully!


In [30]:
print("Original ResNet50 FC Layer:")
print(resnet.fc)

Original ResNet50 FC Layer:
Linear(in_features=2048, out_features=1000, bias=True)


In [31]:
import torch.nn as nn

# Replace the original ImageNet classifier
resnet.fc = nn.Linear(
    in_features=resnet.fc.in_features,
    out_features=4
)

print("Modified ResNet50 FC Layer:")
print(resnet.fc)

Modified ResNet50 FC Layer:
Linear(in_features=2048, out_features=4, bias=True)


In [32]:
# Freeze all EfficientNet-B0 parameters
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze the final classifier
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

print("EfficientNet-B0 layers frozen.")
print("Classifier remains trainable.")

EfficientNet-B0 layers frozen.
Classifier remains trainable.


In [33]:
# Freeze all ResNet50 parameters
for param in resnet.parameters():
    param.requires_grad = False

# Unfreeze the final fully connected layer
for param in resnet.fc.parameters():
    param.requires_grad = True

print("ResNet50 layers frozen.")
print("FC layer remains trainable.")

ResNet50 layers frozen.
FC layer remains trainable.


In [34]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable

    print("Total parameters    :", f"{total:,}")
    print("Trainable parameters:", f"{trainable:,}")
    print("Frozen parameters   :", f"{frozen:,}")


print("EfficientNet-B0 Parameter Verification")
print("=" * 50)
count_parameters(efficientnet)

print("\nResNet50 Parameter Verification")
print("=" * 50)
count_parameters(resnet)

EfficientNet-B0 Parameter Verification
Total parameters    : 4,012,672
Trainable parameters: 5,124
Frozen parameters   : 4,007,548

ResNet50 Parameter Verification
Total parameters    : 23,516,228
Trainable parameters: 8,196
Frozen parameters   : 23,508,032


In [35]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

print("Loss function:")
print(criterion)

Loss function:
CrossEntropyLoss()


In [36]:
import torch.optim as optim

# EfficientNet-B0 optimizer
efficientnet_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)

# ResNet50 optimizer
resnet_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()),
    lr=0.001
)

print("Optimizers created successfully!")
print("=" * 50)

print("EfficientNet-B0 Optimizer:")
print(efficientnet_optimizer)

print("\nResNet50 Optimizer:")
print(resnet_optimizer)

Optimizers created successfully!
EfficientNet-B0 Optimizer:
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

ResNet50 Optimizer:
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [39]:
# Move ResNet50 to the same device as the input
resnet = resnet.to(device)

print("ResNet50 device:", next(resnet.parameters()).device)

ResNet50 device: cuda:0


In [40]:
# Get a fresh batch
images, labels = next(iter(train_loader))

# Move images to GPU
images = images.to(device)

print("Final Model Verification")
print("=" * 50)

# EfficientNet-B0
efficientnet.eval()

with torch.no_grad():
    efficientnet_outputs = efficientnet(images)

print("EfficientNet-B0")
print("Input shape :", images.shape)
print("Output shape:", efficientnet_outputs.shape)
print("Output dtype:", efficientnet_outputs.dtype)

print("\n" + "=" * 50)

# ResNet50
resnet.eval()

with torch.no_grad():
    resnet_outputs = resnet(images)

print("ResNet50")
print("Input shape :", images.shape)
print("Output shape:", resnet_outputs.shape)
print("Output dtype:", resnet_outputs.dtype)

Final Model Verification
EfficientNet-B0
Input shape : torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 4])
Output dtype: torch.float32

ResNet50
Input shape : torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 4])
Output dtype: torch.float32


In [41]:
print("MediScan — Day 3 Transfer Learning")
print("=" * 60)

print("\nDataset")
print("-" * 60)
print("Training samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Testing samples    :", len(test_dataset))
print("Number of classes  :", 4)

print("\nClass Mapping")
print("-" * 60)
for class_name, class_id in class_to_idx.items():
    print(f"{class_id} → {class_name}")

print("\nModels")
print("-" * 60)
print("EfficientNet-B0    : Pretrained + 4-class classifier")
print("ResNet50           : Pretrained + 4-class FC layer")

print("\nTransfer Learning")
print("-" * 60)
print("EfficientNet trainable parameters:",
      sum(p.numel() for p in efficientnet.parameters() if p.requires_grad))
print("ResNet50 trainable parameters    :",
      sum(p.numel() for p in resnet.parameters() if p.requires_grad))

print("\nLoss Function")
print("-" * 60)
print("CrossEntropyLoss")

print("\nOptimizers")
print("-" * 60)
print("EfficientNet-B0 : Adam, lr=0.001")
print("ResNet50        : Adam, lr=0.001")

print("\nFinal Output Verification")
print("-" * 60)
print("EfficientNet-B0 output:", efficientnet_outputs.shape)
print("ResNet50 output       :", resnet_outputs.shape)

print("\n" + "=" * 60)
print("DAY 3 COMPLETED SUCCESSFULLY!")
print("=" * 60)

MediScan — Day 3 Transfer Learning

Dataset
------------------------------------------------------------
Training samples   : 5040
Validation samples : 1080
Testing samples    : 1080
Number of classes  : 4

Class Mapping
------------------------------------------------------------


NameError: name 'class_to_idx' is not defined

In [42]:
print("MediScan — Day 3 Transfer Learning")
print("=" * 60)

print("\nDataset")
print("-" * 60)
print("Training samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Testing samples    :", len(test_dataset))
print("Number of classes  :", 4)

print("\nClass Mapping")
print("-" * 60)

class_mapping = {
    0: "glioma",
    1: "meningioma",
    2: "notumor",
    3: "pituitary"
}

for class_id, class_name in class_mapping.items():
    print(f"{class_id} → {class_name}")

print("\nModels")
print("-" * 60)
print("EfficientNet-B0    : Pretrained + 4-class classifier")
print("ResNet50           : Pretrained + 4-class FC layer")

print("\nTransfer Learning")
print("-" * 60)

efficientnet_trainable = sum(
    p.numel() for p in efficientnet.parameters()
    if p.requires_grad
)

resnet_trainable = sum(
    p.numel() for p in resnet.parameters()
    if p.requires_grad
)

print("EfficientNet trainable parameters:", efficientnet_trainable)
print("ResNet50 trainable parameters    :", resnet_trainable)

print("\nLoss Function")
print("-" * 60)
print("CrossEntropyLoss")

print("\nOptimizers")
print("-" * 60)
print("EfficientNet-B0 : Adam, lr=0.001")
print("ResNet50        : Adam, lr=0.001")

print("\nFinal Output Verification")
print("-" * 60)
print("EfficientNet-B0 output:", efficientnet_outputs.shape)
print("ResNet50 output       :", resnet_outputs.shape)

print("\n" + "=" * 60)
print("DAY 3 COMPLETED SUCCESSFULLY!")
print("=" * 60)

MediScan — Day 3 Transfer Learning

Dataset
------------------------------------------------------------
Training samples   : 5040
Validation samples : 1080
Testing samples    : 1080
Number of classes  : 4

Class Mapping
------------------------------------------------------------
0 → glioma
1 → meningioma
2 → notumor
3 → pituitary

Models
------------------------------------------------------------
EfficientNet-B0    : Pretrained + 4-class classifier
ResNet50           : Pretrained + 4-class FC layer

Transfer Learning
------------------------------------------------------------
EfficientNet trainable parameters: 5124
ResNet50 trainable parameters    : 8196

Loss Function
------------------------------------------------------------
CrossEntropyLoss

Optimizers
------------------------------------------------------------
EfficientNet-B0 : Adam, lr=0.001
ResNet50        : Adam, lr=0.001

Final Output Verification
------------------------------------------------------------
EfficientNet-